# Investigating mopadis diffusion encoder

In [1]:
import os
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from huggingface_hub import hf_hub_download, login
from torchvision import transforms
import h5py

from mopadi.configs.templates import tcga_brca_autoenc
from mopadi.utils.encode import *

os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
# download model files
autoenc_model_path = hf_hub_download(
    repo_id="KatherLab/MoPaDi",
    filename="brca_512_model/autoenc.ckpt",
)
print(f"Autoencoder's checkpoint downloaded to: {autoenc_model_path}")

Autoencoder's checkpoint downloaded to: /home/mala059b/.cache/huggingface/hub/models--KatherLab--MoPaDi/snapshots/5d8e775e24473c5d8f4c0c57fd5c865c3c2a4aab/brca_512_model/autoenc.ckpt


In [4]:
data_dir = "/data/horse/ws/mala059b-rna2wsi/data/TCGA/"

In [5]:
z = zipfile.ZipFile(os.path.join(data_dir, [f for f in os.listdir(data_dir) if f.lower().endswith('.zip')][0])); print(f"ZIP size: {os.path.getsize(z.filename)} bytes\nFiles: {len(z.namelist())}\nFirst file: {z.infolist()[0].filename} ({z.infolist()[0].file_size} bytes)")

ZIP size: 1164644379 bytes
Files: 2700
First file: tiler_params.json (352 bytes)


In [6]:
# pleae tell me about one png file the dimensions
with z.open([f for f in z.namelist() if f.lower().endswith('.png')][535]) as img_file:
    img = Image.open(img_file).convert("RGB")

In [7]:
diffusion_encoder = ImageEncoder(
        tcga_brca_autoenc(), 
        autoenc_path="split_ckpts/diffusion_without_encoder.ckpt", 
        feat_extractor = None,
        device="cuda:0"
)

Seed set to 0


RuntimeError: No CUDA GPUs are available

In [ ]:
# load previously extracted features with mopadi1 feature encoder for all tiles of the chosen patient
with h5py.File(f"/mnt/bulk-saturn/blablas.h5", "r") as hdf_file:
    print(list(hdf_file.keys()))
    features = torch.from_numpy(hdf_file['feats'][:])
    coords = torch.from_numpy(hdf_file['coords'][:])

In [ ]:
len(coords)

In [ ]:
len(features)

In [ ]:
# find at which index the tile's we loaded here features are stored
target = torch.tensor([22016, 11776], device=coords.device)
idx = (coords == target).all(dim=1).nonzero(as_tuple=True)[0]
idx

In [ ]:
# we need to transform the image first, dataloader as in dataset.py does this automatically, byt we loaded a raw img here
transform_list = [transforms.Resize(size=512, interpolation=transforms.InterpolationMode.BILINEAR),
                  transforms.ToTensor(),
                  transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
]

transform = transforms.Compose(transform_list)

In [ ]:
tensor_img = transform(img).unsqueeze(0).to("cuda:0")
tensor_img.size()

In [ ]:
xT = diffusion_encoder.encode_to_noise(tensor_img, features[idx].to("cuda:0"))b

In [ ]:
decoded = diffusion_encoder.decode_image(xT, features[idx].to("cuda:0"))

In [ ]:
dec_np  = decoded.squeeze().float().cpu().permute(1, 2, 0)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img)
axes[0].set_title("Original")
axes[0].axis("off")

axes[1].imshow(dec_np)
axes[1].set_title("Decoded")
axes[1].axis("off")

plt.tight_layout()
plt.show()